In [14]:
import os

In [15]:
%pwd

'C:\\Users\\jeeva\\Desktop\\MLOPS-Project'

In [16]:
os.chdir("../")

In [17]:
%pwd

'C:\\Users\\jeeva\\Desktop'

In [18]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    target_column:str

In [19]:
from src.mlops_project.constants import *
from src.mlops_project.utils.common import read_yaml,create_directories

In [20]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
        root_dir=config.root_dir,
        data_path=config.data_path,
        target_column=config.target_column
        )

        return data_transformation_config

In [21]:
import os
os.getcwd()

'C:\\Users\\jeeva\\Desktop'

In [22]:
import os
os.chdir("C:/Users/jeeva/Desktop/MLOPS-Project")
print(os.getcwd())

C:\Users\jeeva\Desktop\MLOPS-Project


In [23]:
import os
os.listdir("artifacts")

['data_ingestion',
 'data_transformation',
 'data_validation',
 'model_evaluation',
 'model_trainer']

In [24]:
os.listdir("artifacts/model_trainer")

['model.joblib']

In [25]:
import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from src.mlops_project import logger

In [26]:
class DataTransformation:

    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def train_test_splitting(self):

        data = pd.read_csv(self.config.data_path)

        train_df, test_df = train_test_split(
            data,
            test_size=0.25,
            random_state=42
        )

        os.makedirs(self.config.root_dir, exist_ok=True)

        train_df.to_csv(
            os.path.join(self.config.root_dir, "train.csv"),
            index=False
        )

        test_df.to_csv(
            os.path.join(self.config.root_dir, "test.csv"),
            index=False
        )

        logger.info("Data split into train and test sets")
        logger.info(f"Train shape: {train_df.shape}")
        logger.info(f"Test shape: {test_df.shape}")

        return train_df, test_df

    def get_data_transformer_object(self, train_df):

        target_column = self.config.target_column

        input_features = train_df.drop(columns=[target_column])

        numerical_columns = input_features.select_dtypes(
            include=["int64", "float64"]
        ).columns.tolist()

        logger.info(f"Numerical Columns: {numerical_columns}")

        num_pipeline = Pipeline(
            steps=[
                ("scaler", StandardScaler())
            ]
        )

        preprocessor = ColumnTransformer(
            transformers=[
                ("num_pipeline", num_pipeline, numerical_columns)
            ]
        )

        return preprocessor

    def initiate_data_transformation(self):

        train_df, test_df = self.train_test_splitting()

        target_column = self.config.target_column

        preprocessor = self.get_data_transformer_object(train_df)

        input_train = train_df.drop(columns=[target_column])
        target_train = train_df[target_column]

        input_test = test_df.drop(columns=[target_column])
        target_test = test_df[target_column]

        input_train_arr = preprocessor.fit_transform(input_train)
        input_test_arr = preprocessor.transform(input_test)

        # Save preprocessor
        joblib.dump(
            preprocessor,
            os.path.join(self.config.root_dir, "preprocessor.pkl")
        )

        logger.info("Preprocessor saved successfully")

        return (
            input_train_arr,
            target_train,
            input_test_arr,
            target_test
        )

In [27]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)

    data_transformation.initiate_data_transformation()
except Exception as e:
    raise e

[2026-02-25 11:43:15,517:INFO:common:yaml file:config\config.yaml loaded successfully]
[2026-02-25 11:43:15,524:INFO:common:yaml file:params.yaml loaded successfully]
[2026-02-25 11:43:15,531:INFO:common:yaml file:schema.yaml loaded successfully]
[2026-02-25 11:43:15,536:INFO:common:Created Directory at:artifacts]
[2026-02-25 11:43:15,539:INFO:common:Created Directory at:artifacts/data_transformation]
[2026-02-25 11:43:15,612:INFO:96828997:Data split into train and test sets]
[2026-02-25 11:43:15,619:INFO:96828997:Train shape: (1199, 12)]
[2026-02-25 11:43:15,623:INFO:96828997:Test shape: (400, 12)]
[2026-02-25 11:43:15,639:INFO:96828997:Numerical Columns: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']]


[2026-02-25 11:43:15,666:INFO:96828997:Preprocessor saved successfully]
